# Customer Lifetime Value (CLTV) Analysis

## Objective

The objective of this notebook is to estimate the historical Customer Lifetime Value (CLTV) of customers using transactional purchasing behavior.

The analysis includes:

- Revenue feature engineering
- Customer purchase frequency
- Average order value
- Customer lifespan estimation
- Historical CLTV calculation
- Customer segmentation
- Executive visualizations

The outputs generated in this notebook will support executive reporting and Power BI dashboard development.

In [1]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
# ==========================================
# Project Paths
# ==========================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
VISUALS = PROJECT_ROOT / "visuals"

In [6]:
# ==========================================
# Load Clean Dataset
# ==========================================

df = pd.read_csv(
    PROCESSED_DATA / "online_retail_clean.csv",
    parse_dates=["InvoiceDate"]
)

print(df.shape)
df.head()

(779425, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [7]:
# ==========================================
# Data Validation
# ==========================================

print("Shape:", df.shape)
print("Missing Customer IDs:", df["Customer ID"].isna().sum())
print("Negative Quantity:", (df["Quantity"] < 0).sum())
print("Invalid Prices:", (df["Price"] <= 0).sum())
print("Cancelled Invoices:", df["Invoice"].astype(str).str.startswith("C").sum())

Shape: (779425, 8)
Missing Customer IDs: 0
Negative Quantity: 0
Invalid Prices: 0
Cancelled Invoices: 0


In [8]:
# ==========================================
# Create Revenue Feature
# ==========================================

df["Revenue"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.40
1,12,6.75,81.00
2,12,6.75,81.00
3,48,2.10,100.80
4,24,1.25,30.00


In [9]:
# ==========================================
# Revenue Summary
# ==========================================

df["Revenue"].describe().round(2)

count   779,425.00
mean         22.29
std         227.43
min           0.00
25%           4.95
50%          12.48
75%          19.80
max     168,469.60
Name: Revenue, dtype: float64

In [10]:
# ==========================================
# Customer-Level Summary
# ==========================================

customer_summary = (
    df.groupby("Customer ID")
      .agg(
          TotalRevenue=("Revenue", "sum"),
          TotalOrders=("Invoice", "nunique"),
          TotalTransactions=("Invoice", "count"),
          TotalProducts=("Quantity", "sum"),
          FirstPurchase=("InvoiceDate", "min"),
          LastPurchase=("InvoiceDate", "max")
      )
      .reset_index()
)

customer_summary.head()

,Customer ID,TotalRevenue,TotalOrders,TotalTransactions,TotalProducts,FirstPurchase,LastPurchase
0,"12,346.00","77,556.46",12,34,74285,2009-12-14 08:34:00,2011-01-18 10:01:00
1,"12,347.00","4,921.53",8,222,2967,2010-10-31 14:20:00,2011-12-07 15:52:00
2,"12,348.00","2,019.40",5,51,2714,2010-09-27 14:59:00,2011-09-25 13:13:00
3,"12,349.00","4,428.69",4,175,1624,2010-04-29 13:20:00,2011-11-21 09:51:00
4,"12,350.00",334.40,1,17,197,2011-02-02 16:01:00,2011-02-02 16:01:00


In [11]:
print(customer_summary.shape)

customer_summary.describe().round(2)

(5878, 7)


,Customer ID,TotalRevenue,TotalOrders,TotalTransactions,TotalProducts,FirstPurchase,LastPurchase
count,"5,878.00","5,878.00","5,878.00","5,878.00","5,878.00",5878,5878
mean,"15,315.31","2,955.90",6.29,132.60,"1,788.70",2010-08-22 06:57:33.297039,2011-05-22 16:19:59.469207
min,"12,346.00",2.95,1.00,1.00,1.00,2009-12-01 07:45:00,2009-12-01 09:55:00
25%,"13,833.25",342.28,1.00,20.00,187.00,2010-02-09 14:01:15,2010-11-25 10:24:45
50%,"15,314.50",867.74,3.00,52.00,480.00,2010-06-27 13:31:30,2011-09-05 11:59:00
75%,"16,797.75","2,248.30",7.00,138.00,"1,350.00",2011-01-30 14:30:15,2011-11-14 11:31:15
max,"18,287.00","580,987.04",398.00,"12,435.00","367,193.00",2011-12-09 12:16:00,2011-12-09 12:50:00
std,"1,715.57","14,440.85",13.01,342.19,"8,876.30",NaN,NaN
